# 🧪 高级专家 机考模拟练习二：A/B 实验与统计推断专场

> **🎯 考核目标**：这是你的强项！作为具备 A/B 平台建设经验的候选人，面试官一定会要求你手写核心的底层统计学推导代码。包括：方差缩减（CUPED）、样本量预估、SRM 卡方检验。

---

## 🧪 试题一：手撕 CUPED 方差缩减核心逻辑

**📝 你的任务**：
给你包含实验用户数据的 DataFrame `df_ab`。其中 `pre_value` 是实验前 14 天的购买金额（协变量），`post_value` 是实验期间的购买金额（目标变量）。
要求根据 CUPED 公式：$Y_{cuped} = Y - 	heta \times (X - E[X])$
1. 手动计算出最优的 $\theta$ 权重系数。
2. 生成降噪后的修正指标列 `post_value_cuped`。
3. 计算一下修正前和修正后，指标方差缩小了多少百分比。

In [ ]:
import pandas as pd
import numpy as np

# 生成高度相关的模拟数据
np.random.seed(42)
pre_val = np.random.normal(500, 100, 5000)
post_val = pre_val * 1.05 + np.random.normal(0, 50, 5000) # post 和 pre 高度相关 (~0.9 左右)
df_ab = pd.DataFrame({'pre_value': pre_val, 'post_value': post_val})

# ==================
# 你的代码写在这里：

# 1. 计算 theta
corr_matrix = np.cov(df_ab['pre_value'],df_ab['post_value'])
# corr_matrix
cov_xy = corr_matrix[0,1] # 协变量相关性矩阵
var_x = corr_matrix[0,0] # 实验开始前方差
theta = cov_xy / var_x # 最佳调优系数
mean_x = np.mean(df_ab['pre_value']) #实验开始前大盘均值
# 2. 生成 CUPED 修正指标
df_ab['cuped_post_value'] = df_ab['post_value'] - theta * (df_ab['pre_value'] - mean_x)
# 3. 对比调整前后的 Variance
variance_martic = df_ab['post_value'].var()
variance_cuped_martic = df_ab['cuped_post_value'].var()
mean_martic = df_ab['post_value'].mean()
mean_cuped_martic = df_ab['cuped_post_value'].mean()

variance_percent = 1 - (df_ab['cuped_post_value'].var() / df_ab['post_value'].var() )
# ==================
print(f'cuped前方差：{variance_martic:.4f};cuped后方差:{variance_cuped_martic:.4f}')
print(f'降幅：{variance_percent:.4%}')
print(f'cuped前均值：{mean_martic:.4f};cuped后均值:{mean_cuped_martic:.4f}')
print(f'theta系数{theta:.4f}')
# ---------------------
# 💡 架构师级标准答案：
# cov_matrix = np.cov(df_ab['pre_value'], df_ab['post_value'])
# theta = cov_matrix[0, 1] / np.var(df_ab['pre_value'], ddof=1)
# pre_mean = df_ab['pre_value'].mean()
# df_ab['post_value_cuped'] = df_ab['post_value'] - theta * (df_ab['pre_value'] - pre_mean)
# 降噪比例 = 1 - (df_ab['post_value_cuped'].var() / df_ab['post_value'].var())





cuped前方差：13480.8941;cuped后方差:2552.4494411296105
降幅：81.0662%
cuped前均值：525.0945;cuped后均值:525.0945078179739
theta系数1.0491


## 🧪 试题二：SRM 分流比例不匹配的自动化检测

**📝 你的任务**：
你的实验原本按照 50% vs 50% 的流量均匀下放。但后台数据显示：
实验组分到了 51000 人，对照组分到了 49000 人。
请写一段代码（调用 `scipy.stats` 库），做一次卡方优度拟合检验，判断是否触发 SRM 报警（p < 0.001 即算发生分流异常故障）。

In [18]:
from scipy.stats import chisquare

# 观测人数 (实验组，对照组)
observed_counts = [51000, 49000]

# 期望人数 （对于 50/50，期望各是多少？）
total_users = sum(observed_counts)
expected_counts = [total_users/2,total_users/2]
# ==================
# 你的代码写在这里：
stat,p_value = chisquare(f_obs=observed_counts,f_exp=expected_counts)
print(f'统计量：{stat:.4f}')
print(f'p value:{p_value:.4f}')
print(f'{'通过' if p_value>0.05 else '不通过'}')



# ==================

统计量：40.0000
p value:0.0000
不通过


## 🧪 试题三：Python 实现样本量计算公式

**📝 你的任务**：
对于连续型指标评估均值差异，样本量计算公式为：$n = 2 \times \frac{(Z_{\alpha/2} + Z_{\beta})^2 \times \sigma^2}{MDE^2}$
假设基线方差 $\sigma^2 = 3600$，你希望检测出绝对额提升 (MDE) = 5 元的微观差异。
已知 Alpha=0.05 (Z约为1.96)，Beta=0.2 (Power=80%, Z约为0.84)。
请计算单组需要的最少样本量。

In [ ]:
# 参数定义
variance = 3600
mde = 5
z_alpha_2 = 1.96
z_beta = 0.84

# ==================
# 你的代码写在这里：
effect_size = sms.proportion_effectsize()

# ==================
# 💡 面试官防守要点：如果计算出 n = 22579.2，必须要使用 math.ceil() 向上取整，样本量不能是小数，且宁可多不可少。

